# 🌿 Saffron Authenticity: Real vs Fake — Complete Run

**Run every cell top-to-bottom to get the best results.**  
GPU runtime required: `Runtime → Change runtime type → T4 GPU`.

| Step | What it does | Runtime |
|------|-------------|--------|
| 1 | Install packages & check GPU | ~1 min |
| 2 | Mount Drive (optional) | instant |
| 3 | Build leakage-free 5-fold manifest | ~2 min |
| 4 | 5-fold cross-validation (best settings) | ~3–5 hr GPU |
| 5 | Aggregate & display summary tables | ~1 min |
| 6 | Leave-one-camera-out evaluation | ~1–2 hr GPU |
| 7 | Stress test (degraded images) | ~30 min GPU |
| 8 | Display all results & save to Drive | ~2 min |

> ⚡ **Resumable**: finished folds are skipped automatically. Re-run from the top after a disconnect.

In [ ]:
# ── Step 1: Install packages & check GPU ────────────────────────────────────
import subprocess, sys, os

# Install requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# GPU check
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f"✅ GPU: {r.stdout.strip()}")
else:
    print("⚠ No GPU found. Training will be very slow on CPU.")

import tensorflow as tf
print(f"TensorFlow {tf.__version__}  |  GPUs: {tf.config.list_physical_devices('GPU')}")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print("Step 1 complete ✅")

In [ ]:
# ── Step 2: Mount Google Drive (skip if running locally) ────────────────────
# Edit these two paths to point to your Drive folders.
DRIVE_DATA = "/content/drive/MyDrive/saffron_data"   # folder containing the two .zip files
DRIVE_RUNS = "/content/drive/MyDrive/saffron_runs"   # checkpoint backup folder (created automatically)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print(f"Drive mounted ✅")
    print(f"  Data source : {DRIVE_DATA}")
    print(f"  Runs backup : {DRIVE_RUNS}")
except ModuleNotFoundError:
    print("Not in Colab — Drive mount skipped. Place dataset zips in data/ manually.")
    DRIVE_DATA = "data"
    DRIVE_RUNS = "runs"
print("Step 2 complete ✅")

In [ ]:
# ── Step 3: Build leakage-free 5-fold manifest ──────────────────────────────
# Copies zips from Drive → local disk, extracts, deduplicates, assigns sample-level folds.
import os, glob, shutil, subprocess, sys

LOCAL_DATA = "data"
os.makedirs(LOCAL_DATA, exist_ok=True)

manifest = os.path.join(LOCAL_DATA, "manifest.csv")
if os.path.exists(manifest):
    print(f"manifest.csv already exists — skipping. Delete it to force a fresh run.")
else:
    # Copy zips from Drive to local disk
    zips = sorted(glob.glob(os.path.join(DRIVE_DATA, "*.zip")))
    if len(zips) != 2:
        raise RuntimeError(f"Expected exactly 2 .zip files in {DRIVE_DATA}, found {zips}")
    local_zips = []
    for z in zips:
        dst = os.path.join(LOCAL_DATA, os.path.basename(z))
        if not os.path.exists(dst):
            print(f"  Copying {os.path.basename(z)} ...", end="", flush=True)
            shutil.copy2(z, dst); print(" done")
        local_zips.append(dst)

    proc = subprocess.Popen(
        [sys.executable, "scripts/prepare_dataset.py",
         "--src", *local_zips, "--data", LOCAL_DATA, "--folds", "5"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"}
    )
    for line in proc.stdout: print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0: raise RuntimeError("prepare_dataset.py failed")

import pandas as pd
m = pd.read_csv(manifest)
print(f"\nManifest: {len(m)} images | {m.group.nunique()} samples | "
      f"fake {(m.label==1).sum()} | real {(m.label==0).sum()}")
print(m.groupby('fold').agg(images=('path','size'), samples=('group','nunique'), fake_frac=('label','mean')).round(3).to_string())
print("Step 3 complete ✅")

In [ ]:
# ── Step 4: 5-fold cross-validation (best settings) ─────────────────────────
# img-size 256, ImageNet weights, 3 warm + 25 fine-tune epochs, MC samples 16
# Finished folds are skipped automatically → safe to re-run after a disconnect
import os, subprocess, sys, time, shutil

LOCAL_DATA = "data"
LOCAL_RUNS = "runs/cv1"
os.makedirs(LOCAL_RUNS, exist_ok=True)

# Resume from Drive if previous run exists
drive_cv1 = os.path.join(DRIVE_RUNS, "cv1") if 'DRIVE_RUNS' in dir() else ""
if drive_cv1 and os.path.isdir(drive_cv1):
    print("Syncing previous results from Drive...")
    subprocess.run(["rsync", "-a", "--exclude=*.weights.h5",
                    drive_cv1 + "/", LOCAL_RUNS + "/"], capture_output=True)
    done = [d for d in os.listdir(LOCAL_RUNS)
            if d.startswith("fold") and os.path.exists(f"{LOCAL_RUNS}/{d}/cache_test.npz")]
    print(f"  Folds already complete: {sorted(done)}")

def run_streaming(cmd, env_extra=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"}
    if env_extra: env.update(env_extra)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout: print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0: raise subprocess.CalledProcessError(proc.returncode, cmd)

t0 = time.time()
run_streaming([
    sys.executable, "scripts/run_cv.py",
    "--manifest", f"{LOCAL_DATA}/manifest.csv",
    "--out",      LOCAL_RUNS,
    "--img-size", "256",
    "--batch",    "16",
    "--epochs-warm", "3",
    "--epochs-ft",   "25",
    "--mc-samples",  "16",
    "--pop",  "20",
    "--iters", "25",
    "--seed",  "42",
])
print(f"\nTotal training time: {(time.time()-t0)/60:.1f} min")

# Sync to Drive
if drive_cv1:
    os.makedirs(drive_cv1, exist_ok=True)
    subprocess.run(["rsync", "-a", "--exclude=*.weights.h5",
                    LOCAL_RUNS + "/", drive_cv1 + "/"], capture_output=True)
    print(f"Results synced to Drive: {drive_cv1}")
print("Step 4 complete ✅")

In [ ]:
# ── Step 5: Aggregate & display summary tables ───────────────────────────────
import os, sys, subprocess, pandas as pd
from IPython.display import display

LOCAL_DATA = "data"; LOCAL_RUNS = "runs/cv1"

def run_streaming(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"})
    for line in proc.stdout: print(line, end="", flush=True)
    proc.wait()

run_streaming([sys.executable, "scripts/run_cv.py",
               "--manifest", f"{LOCAL_DATA}/manifest.csv",
               "--out", LOCAL_RUNS, "--aggregate-only"])

KEYS = ["acc", "f1_fake", "recall_fake", "specificity", "auc"]
for level, fname in [("Sample level (★ primary)", "summary_sample.csv"),
                     ("Image level", "summary_image.csv"),
                     ("Pooled OOF sample", "pooled_oof_sample_level.csv")]:
    fp = f"{LOCAL_RUNS}/{fname}"
    if not os.path.exists(fp): print(f"⚠ {fname} not found"); continue
    df = pd.read_csv(fp, index_col=0)
    cols = [c for c in KEYS if c in df.columns]
    print(f"\n=== {level} ===")
    display(df[cols].style.highlight_max(axis=0, color="lightgreen").format(
        lambda x: f"{x:.4f}" if isinstance(x, float) else x))
print("Step 5 complete ✅")

In [ ]:
# ── Step 6: Leave-one-camera-out (LOCO) evaluation ──────────────────────────
# Trains on one phone, tests on the other — the strongest cross-device test.
import os, sys, subprocess, glob, shutil, pandas as pd
from IPython.display import display

LOCAL_DATA = "data"; LOCAL_RUNS = "runs/cv1"

def run_streaming(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"})
    for line in proc.stdout: print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0: raise subprocess.CalledProcessError(proc.returncode, cmd)

manifest = f"{LOCAL_DATA}/manifest.csv"

# Generate LOCO manifests
print("Generating LOCO manifests...")
run_streaming([sys.executable, "scripts/make_loco_manifests.py", "--manifest", manifest])

loco_manifests = sorted(glob.glob(f"{LOCAL_DATA}/manifest_loco_*_to_*.csv"))
if not loco_manifests:
    print("⚠ No LOCO manifests generated — your manifest may not have ≥2 usable cameras.")
else:
    PROPOSED = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
    comparison = []
    cv_summary = f"{LOCAL_RUNS}/summary_sample.csv"
    if os.path.exists(cv_summary):
        cv = pd.read_csv(cv_summary, index_col=0)
        if PROPOSED in cv.index:
            row = cv.loc[PROPOSED]
            comparison.append({"Evaluation": "Standard CV", "AUC": row.get("auc", "—"),
                               "F1 (fake)": row.get("f1_fake", "—"), "Accuracy": row.get("acc", "—")})
    for mf in loco_manifests:
        name  = os.path.splitext(os.path.basename(mf))[0]
        parts = name.replace("manifest_loco_", "").split("_to_")
        out_dir = f"runs/loco_{parts[0]}_to_{parts[1]}"
        print(f"\n{'='*55}\nLOCO: {name}\n{'='*55}")
        run_streaming([
            sys.executable, "scripts/run_cv.py",
            "--manifest", mf, "--out", out_dir,
            "--folds", "0", "--img-size", "256",
            "--pop", "20", "--iters", "25", "--seed", "42",
        ])
        summary = f"{out_dir}/summary_sample.csv"
        if not os.path.exists(summary): raise FileNotFoundError(summary)
        df = pd.read_csv(summary, index_col=0)
        if PROPOSED not in df.index: raise KeyError(f"PROPOSED method missing from {summary}")
        row = df.loc[PROPOSED]
        comparison.append({"Evaluation": os.path.basename(out_dir), "AUC": row.get("auc", "—"),
                           "F1 (fake)": row.get("f1_fake", "—"), "Accuracy": row.get("acc", "—")})
        if 'DRIVE_RUNS' in dir() and DRIVE_RUNS:
            drive_out = os.path.join(DRIVE_RUNS, os.path.basename(out_dir))
            os.makedirs(drive_out, exist_ok=True)
            drive_summary = os.path.join(drive_out, "summary_sample.csv")
            if os.path.abspath(summary) != os.path.abspath(drive_summary):
                shutil.copy2(summary, drive_summary)
                print(f"LOCO summary synced to Drive: {drive_out}")
    if comparison:
        print("\nLOCO vs standard CV — proposed model, sample level")
        display(pd.DataFrame(comparison).set_index("Evaluation"))
print("Step 6 complete ✅")

In [ ]:
# ── Step 7: Stress test (degraded test images) ───────────────────────────────
# Needs final.weights.h5 in memory (set SAVE_WEIGHTS_TO_DRIVE=True to persist)
import os, sys, subprocess, pandas as pd
from IPython.display import display

LOCAL_DATA = "data"; LOCAL_RUNS = "runs/cv1"

have_weights = any(
    os.path.exists(f"{LOCAL_RUNS}/fold{k}/final.weights.h5") or
    os.path.exists(f"{LOCAL_RUNS}/fold{k}/best.weights.h5")
    for k in range(5)
)
if not have_weights:
    print("⚠ No saved weights found. Stress test requires the Colab session to still be alive\n"
          "  OR SAVE_WEIGHTS_TO_DRIVE=True set before training. Skipping.")
else:
    def run_streaming(cmd):
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1,
                                 env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2"})
        for line in proc.stdout: print(line, end="", flush=True)
        proc.wait()
        if proc.returncode != 0: raise subprocess.CalledProcessError(proc.returncode, cmd)

    run_streaming([
        sys.executable, "scripts/stress_test.py",
        "--manifest", f"{LOCAL_DATA}/manifest.csv",
        "--run", LOCAL_RUNS,
    ])

    # Display styled metric pivots and save a compact heatmap figure
    import matplotlib.pyplot as plt
    R = pd.read_csv(f"{LOCAL_RUNS}/stress_results.csv")
    PERTURBS = ["clean","blur","jpeg","noise","lowres","gray","hue","dark"]
    for metric in ("acc", "auc"):
        T = R[R.level=="sample"].pivot(index="method", columns="perturbation", values=metric)
        T = T[[c for c in PERTURBS if c in T.columns]]
        print(f"\n=== {metric.upper()} by perturbation (sample level) ===")
        display(T.round(3).style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.3f}"))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric in zip(axes, ("acc", "auc")):
        T = R[R.level=="sample"].pivot(index="method", columns="perturbation", values=metric)
        T = T[[c for c in PERTURBS if c in T.columns]]
        image = ax.imshow(T.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
        ax.set_title(metric.upper()); ax.set_xticks(range(len(T.columns)))
        ax.set_xticklabels(T.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(T.index))); ax.set_yticklabels([m[:35] for m in T.index], fontsize=7)
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout(); fig.savefig(f"{LOCAL_RUNS}/stress_test_heatmap.png", dpi=140, bbox_inches="tight")
    plt.show(); print(f"Heatmap saved: {LOCAL_RUNS}/stress_test_heatmap.png")
    print("\nKey insight: if 'gray' drops to ~50%, colour is the main cue.")
print("Step 7 complete ✅")

In [ ]:
# ── Step 8: Display all results & save to Drive ──────────────────────────────
import os, shutil, time, glob, pandas as pd
from IPython.display import display, HTML

LOCAL_RUNS = "runs/cv1"
PROPOSED   = "PROPOSED: Wombat-OA + weighted beam + log-prob fusion"
BASELINE   = "colour-statistics random forest (baseline)"
summary_fp = f"{LOCAL_RUNS}/summary_sample.csv"
if os.path.exists(summary_fp):
    print("Sample-level CV summary (mean ± std):")
    display(pd.read_csv(summary_fp, index_col=0).style)
stress_fp = f"{LOCAL_RUNS}/stress_results.csv"
if os.path.exists(stress_fp):
    print("Stress-test results:")
    display(pd.read_csv(stress_fp).head(40))

# ── Headline numbers ─────────────────────────────────────────────────────────
mean_fp = f"{LOCAL_RUNS}/summary_sample_mean.csv"
if os.path.exists(mean_fp):
    mean_df = pd.read_csv(mean_fp, index_col=0)
    prop_auc  = mean_df.loc[PROPOSED, "auc"]  if PROPOSED  in mean_df.index else float("nan")
    base_auc  = mean_df.loc[BASELINE, "auc"]  if BASELINE  in mean_df.index else float("nan")
    prop_f1   = mean_df.loc[PROPOSED, "f1_fake"] if PROPOSED in mean_df.index else float("nan")
    beat = "✅ Beats" if prop_auc > base_auc else "⚠  Does NOT beat"
    print("=" * 55)
    print("HEADLINE RESULTS  (sample level, 5-fold group CV)")
    print("=" * 55)
    print(f"  Proposed (Wombat-OA) AUC : {prop_auc:.3f}")
    print(f"  Colour baseline AUC      : {base_auc:.3f}")
    print(f"  Proposed F1(fake)        : {prop_f1:.3f}")
    print(f"  {beat} the colour baseline")
    print("=" * 55)

# ── Package & save to Drive ───────────────────────────────────────────────────
ts = time.strftime("%Y%m%d_%H%M")
tmp = f"runs/results_tmp_{ts}"; os.makedirs(tmp, exist_ok=True)
result_dirs = [LOCAL_RUNS] + sorted(glob.glob("runs/loco_*"))
for result_dir in result_dirs:
    for dp, _, fns in os.walk(result_dir):
        for fn in fns:
            if os.path.splitext(fn)[1].lower() not in (".csv", ".json", ".png"): continue
            src = os.path.join(dp, fn)
            rel = os.path.join(os.path.basename(result_dir), os.path.relpath(src, result_dir))
            dst = os.path.join(tmp, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
zip_path = shutil.make_archive(f"runs/saffron_results_{ts}", "zip", tmp)
shutil.rmtree(tmp)
print(f"Results zip: {zip_path}")

if 'DRIVE_RUNS' in dir() and DRIVE_RUNS:
    os.makedirs(DRIVE_RUNS, exist_ok=True)
    shutil.copy2(zip_path, os.path.join(DRIVE_RUNS, os.path.basename(zip_path)))
    print(f"Saved to Drive: {DRIVE_RUNS}")

try:
    from google.colab import files as _cf; _cf.download(zip_path)
    print("Download triggered ✅")
except Exception:
    print(f"Find zip at: {zip_path}")
print("Step 8 complete ✅ — All done!")